In [1]:
import warnings
warnings.filterwarnings('ignore')

# 교차 검증과 그리드 서치
- 머신러닝을 사용할 때 모델의 정확도를 측정하기 위해 반드시 사용해야 하는 방법.
- 딥러닝시에는 데이터의 크기가 크므로 이 방법은 사용할 필요가 없다.

In [1]:
import pandas as pd
wine = pd.read_csv("../Data/wine.csv")
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [2]:
# Feature, Target
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()

# 검증 세트 추가

In [3]:
# 전체 세트중 훈련세트와 테스트 세트를 8:2의 기준으로 분리한다.
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)

In [4]:
# 훈련세트 중 훈련세트와 검증세트를 '다시' 8:2의 기준으로 분리한다.
sub_input, val_input, sub_target, val_target = train_test_split(
    train_input, train_target, test_size=0.2, random_state=42
)

In [5]:
# 훈련세트, 검증세트, 테스트 세트의 크기 구하기
print("훈련세트 :", sub_input.shape)
print("검증세트 :", val_input.shape)
print("테스트 세트 :", test_input.shape)

훈련세트 : (4157, 3)
검증세트 : (1040, 3)
테스트 세트 : (1300, 3)


In [6]:
# 훈련세트와 검증세트를 결정트리로 모델 만들기

from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print("Train score :", dt.score(sub_input, sub_target))
print("valid score :", dt.score(val_input, val_target))

Train score : 0.9971133028626413
valid score : 0.864423076923077


---
# 교차검증

In [7]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
scores

{'fit_time': array([0.00664186, 0.00664115, 0.00737309, 0.00621009, 0.00647783]),
 'score_time': array([0.00091004, 0.00067306, 0.00120902, 0.00068569, 0.00072217]),
 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}

In [8]:
import numpy as np

np.mean(scores['test_score'])

0.855300214703487

---
# KFold : 분할기를 사용한 교차 검증

In [12]:
from sklearn.model_selection import StratifiedKFold
splitter = StratifiedKFold(n_splits=10) # 기본 n_splits : 5
scores = cross_validate(dt, train_input, train_target, cv=splitter)
scores

{'fit_time': array([0.00722814, 0.00762105, 0.00735188, 0.00689006, 0.00797701,
        0.00815797, 0.00791979, 0.00721097, 0.00746012, 0.00656605]),
 'score_time': array([0.00060296, 0.00069904, 0.00057602, 0.00053   , 0.00069499,
        0.00071311, 0.00054026, 0.0018754 , 0.00062704, 0.00051475]),
 'test_score': array([0.84807692, 0.85769231, 0.875     , 0.86730769, 0.88461538,
        0.87692308, 0.875     , 0.86705202, 0.8477842 , 0.81695568])}

In [13]:
np.mean(scores['test_score'])

0.8616407292129834

In [21]:
# KFold의 Fold를 10개로 나누어서 교차검증       
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) # 기본 n_splits : 5
scores = cross_validate(dt, train_input, train_target, cv=splitter)
scores
# np.mean(scores['test_score'])

{'fit_time': array([0.00725698, 0.00730896, 0.00754905, 0.00726295, 0.00750208,
        0.00735021, 0.0070858 , 0.00806189, 0.00751209, 0.00796604]),
 'score_time': array([0.00060797, 0.00058198, 0.00211501, 0.000741  , 0.000525  ,
        0.00054765, 0.00051022, 0.00054216, 0.00091505, 0.00061107]),
 'test_score': array([0.83461538, 0.87884615, 0.85384615, 0.85384615, 0.84615385,
        0.87307692, 0.85961538, 0.85549133, 0.85163776, 0.86705202])}

In [18]:
splitter = StratifiedKFold(n_splits=10) # 기본 n_splits : 5
scores = cross_validate(dt, train_input, train_target, cv=splitter)
np.mean(scores['test_score'])

0.8616407292129834